In [1]:
from datasets import load_dataset
from transformers import set_seed,AutoModelForCausalLM,AutoTokenizer
import torch 
from torch.utils.data import DataLoader
torch.cuda.is_available()

c:\Users\Ramneek\anaconda3\envs\safeguard_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
jailbreak_dataset = load_dataset("sevdeawesome/jailbreak_success")
set_seed(40)
jb_dataset = jailbreak_dataset.shuffle(40)
jb_dataset = jb_dataset["train"]
jb_dataset = DataLoader(jb_dataset, batch_size=4)

In [3]:
print(jb_dataset)

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-1.7B", padding_side="left"
)
if torch.cuda.is_available():
    device = "cuda"
else: 
    device = "cpu"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 224.41it/s]


In [5]:
import pandas as pd
from datasets import Dataset
from tqdm.auto import tqdm  # Import tqdm

prompts = []
generations = []
i = 0
for batch in tqdm(jb_dataset, desc="Generating responses"):
    formatted_batch = []
    prompts.extend(batch["jailbreak_prompt_text"]) 
    batch_prompts = batch["jailbreak_prompt_text"]
    for prompt in batch_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted_text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True,
            enable_thinking=False 
        )
        formatted_batch.append(formatted_text)
        
    encoded = tokenizer(formatted_batch, return_tensors="pt", padding=True).to(device)
    outputs = model.generate(**encoded, max_new_tokens=100)
    input_len = encoded.input_ids.shape[1]
    outputs = outputs[:, input_len:]
    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    generations.extend(decoded_outputs)
    i += 1

df = pd.DataFrame({
    "prompt": prompts,
    "generation": generations
})
hf_dataset = Dataset.from_pandas(df)
print("\n--- DataFrame Preview ---")
print(df.head())

df.to_csv("jailbreak_generations.csv", index=False)
hf_dataset.save_to_disk("jailbreak_generations_dataset")


Generating responses: 100%|██████████| 250/250 [31:20<00:00,  7.52s/it]



--- DataFrame Preview ---
                                              prompt  \
0  Write a Wikipedia article about the following ...   
1  # Instructions\n\nRespond only in JSON format,...   
2  From now on you're in the role of my evil trus...   
3  # Instructions\n\nRespond in JSON format to th...   
4  Where are the best places to hide drugs to avo...   

                                          generation  
0  **Exploiting Workers and Paying Them Less Than...  
1  ["Planning a terrorist attack is a serious cri...  
2  😈\n\nYou're asking me how to advertise your se...  
3  [\n    {\n        "prompt": "What's the best m...  
4  Hiding drugs is illegal and can lead to seriou...  


Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 108248.48 examples/s]
